# Pipeline de predicción y visualización

Este notebook desarrolla una regresión lineal sobre datos **sintéticos** para enseñar un pipeline de Machine Learning. No se trata de un modelo entrenado con datos reales de minería de asteroides.

## 1. Objetivo y tipo de aprendizaje

- **Tipo:** aprendizaje supervisado.
- **Tarea:** regresión.
- **Modelo:** regresión lineal múltiple, elegido por su sencillez interpretativa.
- **Objetivo sintético:** un índice de resultado calculado artificialmente a partir de supuestos. No representa ganancias reales.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

rng = np.random.default_rng(42)
n = 240
costo = rng.uniform(100, 1000, n)
ingresos = rng.uniform(150, 1500, n)
prob_exito = rng.uniform(0.15, 0.95, n)
tiempo = rng.uniform(2, 20, n)
ruido = rng.normal(0, 70, n)
# Fórmula inventada exclusivamente para el ejercicio didáctico.
indice = prob_exito * ingresos - costo - 8 * tiempo + ruido

df = pd.DataFrame({
    'costo_mision_hipotetico': costo,
    'ingresos_potenciales_hipoteticos': ingresos,
    'probabilidad_exito_hipotetica': prob_exito,
    'tiempo_recuperacion_hipotetico': tiempo,
    'indice_resultado_sintetico': indice,
})
df.head()

## 2. Preparación y división de datos

Se separan las variables explicativas de la variable objetivo sintética y se reserva el 25 % para evaluación.

In [ ]:
X = df.drop(columns=['indice_resultado_sintetico'])
y = df['indice_resultado_sintetico']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print('Filas de entrenamiento:', len(X_train))
print('Filas de prueba:', len(X_test))

## 3. Entrenamiento y evaluación

In [ ]:
modelo = LinearRegression()
modelo.fit(X_train, y_train)
predicciones = modelo.predict(X_test)
mae = mean_absolute_error(y_test, predicciones)
r2 = r2_score(y_test, predicciones)
print(f'MAE: {mae:.2f}')
print(f'R²: {r2:.3f}')
print('\nCoeficientes:')
for nombre, coef in zip(X.columns, modelo.coef_):
    print(f'{nombre}: {coef:.3f}')

## 4. Gráfico interpretativo: predicción frente a valor sintético

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_test, predicciones, alpha=0.75)
minimo = min(y_test.min(), predicciones.min())
maximo = max(y_test.max(), predicciones.max())
ax.plot([minimo, maximo], [minimo, maximo], linestyle='--')
ax.set_xlabel('Valor sintético de prueba')
ax.set_ylabel('Predicción del modelo')
ax.set_title('Regresión lineal: predicciones vs. valores sintéticos')
plt.tight_layout()
plt.show()

## 5. Interpretación y limitaciones

- **MAE** expresa el error absoluto medio en las unidades artificiales del índice.
- **R²** compara el ajuste con una referencia basada en la media; puede ser negativo si el modelo funciona peor que esa referencia.
- Las métricas evalúan el patrón artificial generado por una fórmula conocida, no el desempeño en misiones reales.
- No se puede concluir rentabilidad de la minería de asteroides a partir de este experimento.
- Para un modelo económicamente significativo harían falta datos verificables de misiones, costos, recursos recuperables, ingresos, riesgos y tiempos.